# 03. Governance: SageMaker Model Registry and Model Card

1. **Model Package Group (BP-601):** one registry entry for the term deposit model; each retrain becomes a new version.
2. **Register the version (BP-602):** the model artifact, its container, and its evaluation metrics, as *PendingManualApproval*.
3. **Review and approve (BP-603):** a recorded decision based on the metrics; only approved versions get deployed (notebook 04).
4. **Model Card (BP-604):** intended use, limitations and evaluation results, stored in SageMaker next to the model.

Run notebook 02 first: it saves the training job, model artifact and metrics.

In [1]:
# Setup: repository root, config, lab connection and the state saved by earlier notebooks.
from pathlib import Path

import pandas as pd
import sagemaker

from bankpulse_core.aws import lab_boto_session, read_state, s3_uri, update_state
from bankpulse_core.config import load_config

ROOT = Path.cwd() if (Path.cwd() / 'config').is_dir() else Path.cwd().parent
config = load_config(ROOT / 'config' / 'data_prep.yaml')
STATE = Path(config['output']['processed_dir']) / 'sagemaker_state.json'
state = read_state(STATE)

boto_session = lab_boto_session()                       # bankpulse-lab profile; no keys in code
sm_session = sagemaker.Session(boto_session=boto_session)
sm_client = boto_session.client('sagemaker')
bucket, prefix, role = state['bucket'], state['prefix'], state['role']

import json

from botocore.exceptions import ClientError

from bankpulse_core.io import read_json, upload_file, write_json

assert 'metrics_uri' in state, 'Run notebook 02 first'
metrics = read_json(state['metrics_uri'])
comparison = read_json(ROOT / 'reports' / 'model_comparison.json')
lineage = read_json(state['s3']['feature_schema'].rsplit('/feature-schema/', 1)[0] + '/run-metadata/run_metadata.json')
GROUP = 'bankpulse-term-deposit'
print('model artifact:', state['model_data'].split(f'{bucket}/', 1)[1])

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\aishw\AppData\Local\sagemaker\sagemaker\config.yaml


model artifact: bank-campaign-mlops/model-artifacts/bankpulse-xgboost-2026-10-20-19-38-10-728/output/model.tar.gz


## BP-601. Model Package Group

A **model package group** is the registry's folder for one model: every registered version sits under it, with its own approval status.

**Screenshot:** SageMaker console -> Model registry -> the group.

In [2]:
# Create the group once; on a re-run it already exists, which is fine.
try:
    sm_client.create_model_package_group(
        ModelPackageGroupName=GROUP,
        ModelPackageGroupDescription='Predicts whether a bank customer will subscribe to a term deposit '
                                     '(UCI Bank Marketing), to rank customers before a phone campaign.',
    )
    print('created group', GROUP)
except ClientError as error:
    if 'already exists' not in str(error):
        raise
    print('group already exists:', GROUP)

created group bankpulse-term-deposit


## BP-602. Register the model version

A **model package** bundles everything needed to deploy the model again later:
- the inference container image and `model.tar.gz`
- the content types it accepts and returns (CSV)
- the instance types it is meant for
- **model metrics**: a JSON file in SageMaker's model-quality format, shown in the registry's metrics tab
- custom metadata: the decision threshold, the training job, and the git commit of the data pipeline

It is registered as **PendingManualApproval**: nothing can be deployed from it until someone approves it.

**Screenshot:** the version with status **PendingManualApproval**, and its metrics.

In [3]:
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics

test = metrics['test']
cm = test['confusion_matrix']

# Step 1: test metrics in SageMaker's binary-classification model-quality format.
def metric(value):
    return {'value': value, 'standard_deviation': 'NaN'}

model_quality = {'binary_classification_metrics': {
    'auc': metric(test['roc_auc']),
    'precision': metric(test['precision']),
    'recall': metric(test['recall']),
    'f1': metric(test['f1']),
    'confusion_matrix': {'0': {'0': cm['tn'], '1': cm['fp']}, '1': {'0': cm['fn'], '1': cm['tp']}},
}}
quality_path = write_json(model_quality, Path(config['output']['processed_dir']) / 'model_quality.json')
quality_uri = upload_file(quality_path, s3_uri(bucket, prefix, 'reports', 'model_quality.json'))

# Step 2: the deployable model (container + artifact) and its metrics.
model = Model(image_uri=state['image_uri'], model_data=state['model_data'], role=role, sagemaker_session=sm_session)
model_metrics = ModelMetrics(model_statistics=MetricsSource(s3_uri=quality_uri, content_type='application/json'))

# Step 3: register a new version in the group, waiting for approval.
package = model.register(
    model_package_group_name=GROUP,
    content_types=['text/csv'],
    response_types=['text/csv'],
    inference_instances=['ml.m5.large'],
    transform_instances=['ml.m5.large'],
    approval_status='PendingManualApproval',
    model_metrics=model_metrics,
    description=f'XGBoost 1.7-1, test PR-AUC {test["pr_auc"]}, threshold {metrics["threshold"]:.4f}',
    customer_metadata_properties={
        'threshold': str(metrics['threshold']),
        'training_job': state['training_job'],
        'pipeline_git_commit': str(lineage['code']['git_commit']),
        'test_pr_auc': str(test['pr_auc']),
    },
)
package_arn = package.model_package_arn
state = update_state(STATE, model_package_group=GROUP, model_package_arn=package_arn)
details = sm_client.describe_model_package(ModelPackageName=package_arn)
display(pd.Series({'version': details['ModelPackageVersion'], 'status': details['ModelApprovalStatus'],
                   'description': details['ModelPackageDescription']}).to_frame('model package'))

,model package
version,1
status,PendingManualApproval
description,"XGBoost 1.7-1, test PR-AUC 0.4673, threshold 0..."


## BP-603. Review and approve

The approval rule is written down here, not decided by feel. It is a **non-inferiority** rule:
- **Approve** if XGBoost's test PR-AUC is no more than **0.01 below** the logistic baseline's, on the same features.
- **Reject** otherwise, and record why.

Why not "must beat the baseline": on this data the two models are statistically tied on test PR-AUC (a difference of about 0.001). XGBoost is better on validation and on lift, and it's the model the SageMaker pipeline is built around. Note for the record: this margin was set **after** seeing the first results, and the approval description says so.

The decision and its reason are stored on the model package (`ApprovalDescription`), so the registry shows who decided what, and on what evidence.

**Screenshot:** the version with status **Approved**.

In [4]:
MARGIN = 0.01                                     # allowed shortfall in test PR-AUC
pr_auc = comparison['pr_auc']
approved = pr_auc['difference'] >= -MARGIN
reason = (f'Test PR-AUC {pr_auc["candidate"]} vs logistic baseline {pr_auc["baseline"]} '
          f'({pr_auc["difference"]:+.4f}); lift {comparison["lift_top_decile"]["candidate"]}x vs '
          f'{comparison["lift_top_decile"]["baseline"]}x.')

sm_client.update_model_package(
    ModelPackageArn=package_arn,
    ModelApprovalStatus='Approved' if approved else 'Rejected',
    ApprovalDescription=(f'{"Approved" if approved else "Rejected"} under a non-inferiority rule (test PR-AUC at most '
                         f'{MARGIN} below the logistic baseline; margin set after the first results). ' + reason)[:1024],
)
status = sm_client.describe_model_package(ModelPackageName=package_arn)['ModelApprovalStatus']
state = update_state(STATE, approval_status=status)
print(status, '-', reason)

Approved - Test PR-AUC 0.4673 vs logistic baseline 0.4712 (-0.0039); lift 1.239x vs 1.193x.


## BP-604. Model Card

A **Model Card** is the model's documentation, stored in SageMaker: what the model is for, what it must **not** be used for, how it was evaluated, and its known limitations. It is created as a **Draft**, ready for review.

**Screenshots:** the card overview (intended uses, risk rating), and the evaluation and additional-information sections.

In [5]:
CARD = 'bankpulse-term-deposit-v' + str(details['ModelPackageVersion'])
validation = metrics['validation']

def numbers(split_metrics):
    keys = ('pr_auc', 'roc_auc', 'precision', 'recall', 'f1', 'lift_top_decile', 'positive_rate')
    return [{'name': key, 'type': 'number', 'value': split_metrics[key]} for key in keys]

content = {
    'model_overview': {
        'model_description': 'Gradient-boosted trees (built-in SageMaker XGBoost 1.7-1) scoring the probability '
                             'that a customer subscribes to a term deposit after a phone call.',
        'model_owner': 'bankpulse-ds',
        'problem_type': 'Binary Classification',
        'algorithm_type': 'XGBoost',
        'model_artifact': [state['model_data']],
        'inference_environment': {'container_image': [state['image_uri']]},
    },
    'intended_uses': {
        'purpose_of_model': 'Rank customers before a telemarketing campaign so the most likely subscribers are called first.',
        'intended_uses': 'Prioritising call lists for term deposit campaigns, reviewed by campaign staff. '
                         'Not for credit, lending, pricing or any decision about eligibility for a product.',
        'factors_affecting_model_efficiency': 'Economic conditions (Euribor, employment) change over time and '
                                              'drive subscription rates; scores degrade when they drift.',
        'risk_rating': 'Medium',
        'explanations_for_risk_rating': 'Affects who is contacted, not access to products; uses age and job, so '
                                        'selection rates by age band are monitored.',
    },
    'training_details': {
        'training_observations': f'Trained on the oldest 70% of calls (time-ordered split), validated on the next 15%. '
                                 f'Call duration is excluded because it is only known after the call (leakage).',
        'training_job_details': {'training_arn': sm_client.describe_training_job(
            TrainingJobName=state['training_job'])['TrainingJobArn']},
    },
    'evaluation_details': [
        {'name': 'Validation (15%, after training period)', 'evaluation_observation':
            f'Decision threshold {metrics["threshold"]:.4f} chosen here by best F1.',
         'metric_groups': [{'name': 'binary classification', 'metric_data': numbers(validation)}]},
        {'name': 'Test (newest 15%)', 'evaluation_observation':
            'Evaluated once. Subscription rate is 38% vs 6% in training: strong drift, so absolute '
            'scores are not comparable across periods.',
         'metric_groups': [{'name': 'binary classification', 'metric_data': numbers(metrics['test'])}]},
    ],
    'additional_information': {
        'ethical_considerations': 'Age and job are inputs. Selection rates per age band are reported by the '
                                  'monitoring notebook (05); a min/max ratio below 0.8 triggers review.',
        'caveats_and_recommendations': 'Retrain when monitoring shows feature PSI above 0.2 on the economic '
                                       'indicators. Data from one Portuguese bank, 2008-2010.',
        'custom_details': {'model_package_arn': package_arn, 'baseline_test_pr_auc': str(pr_auc['baseline'])},
    },
}

# Create the card, or update it if this notebook already created it.
try:
    sm_client.create_model_card(ModelCardName=CARD, Content=json.dumps(content), ModelCardStatus='Draft')
    print('created model card', CARD)
except ClientError as error:
    if 'already exists' not in str(error):
        raise
    sm_client.update_model_card(ModelCardName=CARD, Content=json.dumps(content))
    print('updated model card', CARD)
state = update_state(STATE, model_card=CARD)
card = sm_client.describe_model_card(ModelCardName=CARD)
display(pd.Series({'name': card['ModelCardName'], 'status': card['ModelCardStatus'],
                   'version': card['ModelCardVersion']}).to_frame('model card'))

created model card bankpulse-term-deposit-v1


,model card
name,bankpulse-term-deposit-v1
status,Draft
version,1
